# Bakehouse Franchises

**Dataset:** `samples.bakehouse.sales_franchises`

**Difficulty:** Easy

**Topics:** groupBy, filter, distinct, count

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — groupBy Patterns

| Function | What it does |
|----------|-------------|
| `df.groupBy(col).count()` | Count rows per group |
| `df.groupBy(col).agg(F.countDistinct("id"))` | Count unique values within each group |
| `df.filter(F.col("col") == value)` | Equality filter |
| `df.filter(F.col("col").isin([v1, v2]))` | Filter to a set of values |
| `df.select("col").distinct()` | Returns unique values for a column |

**Docs:** [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_franchises")

# How many franchises are there total?
print("Total franchises:", df.count())

# What columns does this table have?
df.printSchema()

# Count distinct cities (not countries — the problems ask about countries)
df.select(F.countDistinct("city").alias("unique_cities")).show()

## Problem 1

Count the number of **franchises per country**, sorted from most to fewest.
Load `samples.bakehouse.sales_franchises` and group by country.

**Expected output columns:**
- `country` - country name
- `franchise_count` - number of franchises in that country (sorted descending)

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = spark.read.table("samples.bakehouse.sales_franchises").groupBy("country").agg(
    F.count("*").alias("franchise_count")
).orderBy(F.col("franchise_count").desc())

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'franchise_count' in cols, "Missing column: franchise_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['franchise_count'] for r in result_1.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by franchise_count descending"
assert all(c > 0 for c in counts), "All franchise counts must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find all **distinct franchise size categories** available in the dataset.
Each size should appear exactly once.

**Expected output columns:**
- `size` - unique franchise size label (e.g., Small, Medium, Large)

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = df.select("size").distinct()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'size' in cols, "Missing column: size"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Expected at most 10 distinct sizes, got {cnt}"
sizes = [r['size'] for r in result_2.collect()]
assert len(sizes) == len(set(sizes)), "Size values must be distinct"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Count how many franchises fall into each **size category**.
This helps understand how the franchise portfolio is distributed by size.

**Expected output columns:**
- `size` - franchise size category
- `count` - number of franchises with that size

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = df.groupBy("size").count()
display(result_3)

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'size' in cols, "Missing column: size"
assert 'count' in cols, "Missing column: count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
rows = result_3.collect()
assert all(r['count'] > 0 for r in rows), "All count values must be positive"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

List all franchises located in the **United Kingdom**.
Filter the dataset and return selected identifying columns.

**Expected output columns:**
- `franchiseID` - franchise identifier
- `name` - franchise name
- `city` - city of the franchise
- `size` - franchise size

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = df.filter(F.col("city") == "United Kingdom").select(
    "franchiseID",
    "name",
    "city",
    "size"
)

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'city' in cols, "Missing column: city"
assert 'size' in cols, "Missing column: size"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find franchises where the **district is null** - these may be missing
geographic classification data that needs to be filled in.

**Expected output columns:**
- `franchiseID` - franchise identifier
- `name` - franchise name
- `country` - country of the franchise

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = df.filter(F.col("district").isNull()).select(
    "franchiseID",
    "name",
    "country"
)

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
print(f"Problem 5 passed ✓  ({cnt} rows)")